In [15]:
import tensorflow as tf
import cv2
import imghdr
import os
import tensorflow as tf
import os
import matplotlib.pyplot as plt
from matplotlib.image import imread
import numpy as np
from tensorflow.keras.models import Sequential 
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Flatten, Dropout
from tensorflow.keras.metrics import Precision, Recall, BinaryAccuracy

In [16]:
gpus = tf.config.experimental.list_physical_devices("GPU")
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

In [17]:
data_dir = os.path.expanduser('~/Desktop/Covid Vaccination Records/data')
image_exts = ['jpeg', 'jpg', 'bmp', 'png']

In [18]:
os.listdir(data_dir)

['.DS_Store', 'not', 'trendy']

In [19]:
for image_class in os.listdir(data_dir):
    class_path = os.path.join(data_dir, image_class)
    if not os.path.isdir(class_path):
        continue
    for image in os.listdir(class_path):
        print(image)

1dea8a8979fa3bbc37bb2c97f2638317-5098388810_1_1_0.jpg
Skinny-Pant-Syndrome.jpg
e166126b97845805ab0f2566ba06117c-0358156433_1_1_0.jpg
image34.jpeg
WH689_000_93A-511_009_1_32.jpg
leggings_a728a17e-74fc-4ae1-879b-8d7d17f1ea9a.jpg
image22.jpeg
SKINNY-JEANS-TREND-vogue-business-story.jpg
T1441007-Midnight-1.jpg
100077360_06V_1920x2880.jpg
WH689_000_103-507_098_1_1.jpg
3_2250c58c-50de-46b7-885d-c9bee2183a76.jpg
image18.jpeg
295020227-front-pdp-lse.jpg
image38.jpeg
14ef7f98a0d810c38c1cbad24c7da3a3-5098388610_1_1_0.jpg
8b1df04661187c6b66acbf49437f4aad-0390211428_1_1_0.jpg
bd12a955fe7695db3340af1b32439766-5010156811_1_1_0.jpg
2a1981b2-65af-4b57-864f-737fa908ce69_1.129302e1d4078f6e484e3bc86448c964.jpeg
eafa3bff0a58c9754596d5f97f9bb91c-0358156800_1_1_0.jpg
universa-leggings-cortos-de-talle-alto-y-sujecion-media-con-bolsillos-nH1dGb.png
image43.jpeg
V3Apparelwomensseamlessscrunchworkoutfitnesssquatproofleggingsdefinegreyreverse.jpg
LG2356776-8329-1_577x866.jpg
image14.jpeg
CC-21_dfd82178-f7b5-4777

In [20]:
img = cv2.imread(os.path.join('data','happy','Screenshot 2024-06-30 at 10.59.57.png'))

[ WARN:0@2083.863] global loadsave.cpp:275 findDecoder imread_('data/happy/Screenshot 2024-06-30 at 10.59.57.png'): can't open/read file: check file path/integrity


In [21]:
img.shape

AttributeError: 'NoneType' object has no attribute 'shape'

In [ ]:
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.show()

In [ ]:
for image_class in os.listdir(data_dir):
    for image in os.listdir(os.path.join(data_dir, image_class)):
        image_path = os.path.join(data_dir, image_class)
        try:
            img = cv2.imread(image_path)
            tip = imghdr.what(image_path)
            if tip not in image_exts:
                print('ímage not in ext list {}'.format(image_path))
                os.remove(image_path)

        except Exception as e:
            print('Issue with image {}'.format(image_path))

In [ ]:
for image_class in os.listdir(data_dir):
    class_path = os.path.join(data_dir, image_class)
    if not os.path.isdir(class_path):
        continue  # Skip non-directory items

    for image in os.listdir(class_path):
        image_path = os.path.join(class_path, image)
        try:
            # Read the image using cv2
            img = cv2.imread(image_path)
            if img is None:
                print('Unable to read image: {}'.format(image_path))
                continue
            
            # Check the image type using imghdr
            tip = imghdr.what(image_path)
            if tip not in image_exts:
                print('Image not in expected format: {}'.format(image_path))
                os.remove(image_path)
                print('Removed:', image_path)

        except Exception as e:
            print('Error processing image {}: {}'.format(image_path, str(e)))

In [ ]:
data = tf.keras.utils.image_dataset_from_directory('data')

In [ ]:
data_iterator = data.as_numpy_iterator()

In [ ]:
data_iterator

In [ ]:
batch = data_iterator.next()

In [ ]:
batch[0].shape

In [ ]:
batch[1]

In [ ]:
fig, ax = plt.subplots(ncols=4, figsize=(20,20))
for idx, img in enumerate(batch[0][:4]):
    ax[idx].imshow(img.astype(int))
    ax[idx].title.set_text(batch[1][idx])

#0 is happy
#1 is sad

In [ ]:
scaled = batch[0] / 255

In [ ]:
scaled.max()

In [ ]:
data = data.map(lambda x,y: (x/255, y))

In [ ]:
scaled_iterator = data.as_numpy_iterator()

In [ ]:
scaled_iterator.next()[0].max()

In [ ]:
len(data)

In [ ]:
train_size = int(len(data)*.7)
val_size = int(len(data)*.2)+1
test_size = int(len(data)*.1)+1

In [ ]:
train_size

In [ ]:
val_size

In [ ]:
test_size

In [ ]:
train = data.take(train_size)
val = data.skip(train_size).take(val_size)
test = data.skip(train_size+val_size).take(test_size)

#shuffled data

In [ ]:
model = Sequential()

In [ ]:
model.add(Conv2D(16, (3,3), 1, activation='relu',input_shape=(256,256,3)))
model.add(MaxPooling2D())

model.add(Conv2D(32, (3,3), 1, activation='relu'))
model.add(MaxPooling2D())

model.add(Conv2D(16, (3,3), 1, activation='relu'))
model.add(MaxPooling2D())

model.add(Flatten())

model.add(Dense(256, activation='relu'))
model.add(Dense(1,activation='sigmoid'))

In [ ]:
model.compile('adam', loss=tf.losses.BinaryCrossentropy(), metrics=['accuracy'])

In [ ]:
model.summary()

In [ ]:
#train

logdir = 'logs'

In [ ]:
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=logdir)

In [ ]:
hist = model.fit(train, epochs=20, validation_data=val, callbacks=[tensorboard_callback])

In [ ]:
hist.history

In [ ]:
fig = plt.figure()
plt.plot(hist.history['loss'], color='blue', label='loss')
plt.plot(hist.history['val_loss'], color='red', label='val_loss')
fig.suptitle('Loss', fontsize=20)
plt.legend(loc='upper left')
plt.show()

In [ ]:
fig = plt.figure()
plt.plot(hist.history['accuracy'], color='blue', label='accuracy')
plt.plot(hist.history['val_accuracy'], color='red', label='val_accuracy')
fig.suptitle('Accuracy', fontsize=20)
plt.legend(loc='upper left')
plt.show()

In [ ]:
#evaluate

In [ ]:
pre = Precision()
re = Recall()
acc = BinaryAccuracy()

In [ ]:
len(test)

In [ ]:
for batch in test.as_numpy_iterator():
    x, y = batch
    yhat = model.predict(x)
    pre.update_state(y, yhat)
    re.update_state(y, yhat)
    acc.update_state(y, yhat)

In [ ]:
print(f'Precision:{pre.result().numpy()}, Recall:{re.result().numpy()}, Accuracy:{acc.result().numpy()}')

In [ ]:
#test

In [ ]:
img = cv2.imread('happytest.png')
plt.imshow(cv2.cvtColor(img, cv2. COLOR_BGR2RGB))
plt.show()

In [ ]:
resize = tf.image.resize(img, (256,256))
plt.imshow(resize.numpy().astype(int))
plt.show()

In [ ]:
yhat = model.predict(np.expand_dims(resize/255, 0))

In [ ]:
yhat

In [ ]:
if yhat > 0.5:
    print(f'Predicted class is Happy')
else:
    print(f'Predicted class is Sad')

In [ ]:
from tensorflow.keras.models import load_model

In [ ]:
model.save(os.path.join('models', 'happysadmodel.h5'))

In [ ]:
os.path.join('models', 'happysadmodel.h5')

In [ ]:
new_model = load_model(os.path.join('models', 'happysadmodel.h5'))

In [ ]:
new_model.predict(np.expand_dims(resize/255, 0))